# 03 Features: one row per monotown

This notebook builds the variables used for clustering in notebook 04.
For each of the 247 matched monotowns we calculate:

- population 2025 and population change 2017-2025 (%)
- net migration per 1,000 residents (average of 2021-2024)
- average monthly wage 2025
- jobs per 1,000 residents 2025 and job change 2017-2025 (%)
- industry dependence: share of jobs in mining and manufacturing
- English names for towns, regions and municipality types

Input: the cleaned files from `data/processed/` (notebook 01) and the matched list `monotowns_oktmo.csv` (notebook 02).

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED = Path("../data/processed")
pd.set_option("display.max_columns", 50)

## Load the data and check the loaded files

OKTMO is the Russian code for municipalities (like a postcode for local governments).
We read it as text, not as a number, because some codes start with 0 (e.g. 04712000 for Yeniseysk),
and reading them as numbers would silently drop that zero and break the matching.

In [9]:
indicator_files = ["population", "migration", "wages", "employees", "industry", "profitable"]

data = {
    name: pd.read_csv(PROCESSED / f"{name}.csv", dtype={"oktmo": str, "oktmo_stable": str})
    for name in indicator_files
}

mono = pd.read_csv(PROCESSED / "monotowns_oktmo.csv", dtype={"oktmo": str})

In [12]:
for name, df in data.items():
    years = sorted(df["year"].unique())
    lengths = df["oktmo"].str.len().value_counts().to_dict()
    print(f"{name:11} rows={len(df):>7}  years={years}  oktmo lengths={lengths}")

print()
for name in ["population", "migration", "wages", "employees", "profitable"]:
    dups = data[name].duplicated(["oktmo", "year"]).sum()
    print(f"{name:11} duplicates (oktmo, year): {dups}")

population  rows=  30731  years=[np.int64(2017), np.int64(2025)]  oktmo lengths={8: 30731}
migration   rows=  58560  years=[np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]  oktmo lengths={8: 58560}
wages       rows=   2486  years=[np.int64(2025)]  oktmo lengths={8: 2486}
employees   rows=   4927  years=[np.int64(2017), np.int64(2025)]  oktmo lengths={8: 4927}
industry    rows=  12021  years=[np.int64(2017), np.int64(2025)]  oktmo lengths={8: 12021}
profitable  rows=  13476  years=[np.int64(2024)]  oktmo lengths={8: 13476}

population  duplicates (oktmo, year): 0
migration   duplicates (oktmo, year): 0
wages       duplicates (oktmo, year): 0
employees   duplicates (oktmo, year): 0
profitable  duplicates (oktmo, year): 0


In [14]:
print(mono["note"].value_counts(), "\n")

matched = mono[mono["oktmo"].notna()].copy()
print("matched towns:", len(matched))                          # expected 247
print("duplicated oktmo:", matched["oktmo"].duplicated().sum())  # expected 0
print("oktmo lengths:", matched["oktmo"].str.len().value_counts().to_dict())

note
automatic                                       213
manual correction                                15
excluded: region not in Rosstat data              9
matched via migration table                       9
excluded: closed city (ZATO), no public data      7
district proxy                                    6
city part of merged okrug                         4
excluded: district proxy not representative       1
Name: count, dtype: int64 

matched towns: 247
duplicated oktmo: 0
oktmo lengths: {8: 247}
